## Lab 1: Creating a Financial Product Launch Agent Prototype

### Overview

[Amazon Bedrock AgentCore](https://aws.amazon.com/bedrock/agentcore/) helps you deploy and operate AI agents securely at scale - using any framework and model. It provides you with the capability to move from prototype to production faster.

In this 5-labs tutorial, we will demonstrate the end-to-end journey from prototype to production using a **Financial Product Launch Agent**. For this example we will use [Strands Agents](https://strandsagents.com/latest/), a simple-to-use, code-first framework for building agents and the Anthropic Claude Sonnet 3.7 model from Amazon Bedrock. For your application you can use the framework and model of your choice. It's important to note that the concepts covered here can be applied using other frameworks and models as well.

**Workshop Journey:**
- **Lab 1 (Current)**: Create Agent Prototype - Build a functional product launch agent
- **Lab 2**: Enhance with Memory - Add conversation context and personalization
- **Lab 3**: Scale with Gateway & Identity - Share tools across agents securely
- **Lab 4**: Deploy to Production - Use AgentCore Runtime with observability
- **Lab 5**: Build User Interface - Create a product manager-facing application

In this first lab, we'll build a Financial Product Launch Agent prototype using **real tools** from the project:
- **research_market_data()** - Real-time Tavily API market research
- **real_time_market_research()** - Enhanced financial site analysis
- **browse_web()** - Web browsing for competitive intelligence
- **create_marketing_poster()** - Nova Canvas image generation

### Architecture for Lab 1

*Simple prototype running locally with real API integrations. In subsequent labs, we'll migrate this to AgentCore services with shared tools, persistent memory, and production-grade observability.*

### Prerequisites

* **AWS Account** with appropriate permissions
* **Python 3.10+** installed locally
* **AWS CLI configured** with credentials
* **Anthropic Claude 3.7** enabled on [Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)
* **Tavily API Key** from https://tavily.com
* **Strands Agents** and other libraries installed in the next cells

### Step 1: Install Dependencies and Import Libraries
Before we start, let's install the pre-requisites for this lab

In [ ]:
# Install required packages
%pip install -U -r requirements.txt -q

We can now import the required libraries and initialize our boto3 session

In [ ]:
# Import libraries
import os
import sys
import boto3
from boto3.session import Session

# Add current directory to path for lab_helpers imports
# This works whether you're in the labs directory or parent directory
current_dir = os.getcwd()
print(f"Current directory: {current_dir}")

# If we're in the labs directory, add it to path
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)
    print(f"Added to path: {current_dir}")

# If we're NOT in labs directory, try to add labs subdirectory
if not current_dir.endswith('labs'):
    labs_path = os.path.join(current_dir, 'labs')
    if os.path.exists(labs_path) and labs_path not in sys.path:
        sys.path.insert(0, labs_path)
        print(f"Added to path: {labs_path}")

print(f"Python path: {sys.path[:3]}")

# Import unified market research tool (replaces research_market_data and real_time_market_research)
from lab_helpers.unified_market_research import market_research
from lab_helpers.enhanced_tools import browse_web
from lab_helpers.marketing_tools import create_marketing_poster

print("✅ Successfully imported lab_helpers modules!")

In [ ]:
# Get boto session
boto_session = Session()
region = boto_session.region_name

### Step 2: Implementing Custom Tools

Next, we will implement the 3 tools which will be provided to the Financial Product Launch Agent.

Defining tools in Strands Agent is extremely simple, just add a `@tool` decorator to your function, and provide a description of the tool in the function's docstring. Strands Agents will use the function documentation, typing and arguments to provide context on this tool to your agent.

#### Tool 1: Real-Time Market Research

**Purpose:** Uses Tavily API for real-time market research and competitive analysis with actual web data.

#### Tool 2: Web Browsing

**Purpose:** Browse specific financial websites for detailed competitive intelligence and market data.

### Step 3: Create and Configure the Product Launch Agent

Next, we will create the Financial Product Launch Agent providing a model, the list of tools implemented in the previous step, and with a system prompt.

In [ ]:
from strands import Agent
from strands.models import BedrockModel

SYSTEM_PROMPT = """You are an expert financial product launch assistant helping product managers and marketing teams launch new financial products.
Your role is to:
- Provide real-time market insights using intelligent market research
- Analyze competitive landscape with actual web data
- Support go-to-market strategy development
- Generate marketing materials with Nova Canvas
- Be professional, data-driven, and strategic in your recommendations

You have access to the following tools:
1. market_research() - Intelligent market research that automatically chooses between:
   - Quick research (1-2 sec) for simple rate checks and current data
   - Deep analysis (10-15 sec) for strategic insights and competitive positioning
   Just call market_research() and it will choose the best approach automatically!
2. browse_web() - Browse specific websites for detailed information
3. create_marketing_poster() - Generate marketing materials with Nova Canvas

Always use these tools to get accurate, up-to-date information rather than making assumptions."""

# Initialize the Bedrock model (Anthropic Claude 3.7 Sonnet)
model = BedrockModel(
    model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    temperature=0,
    region_name=region
)

# Create the product launch agent with unified market research tool
agent = Agent(
    model=model,
    tools=[
        market_research,             # Unified intelligent market research
        browse_web,                  # Web browsing capability
        create_marketing_poster      # Nova Canvas image generation
    ],
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Financial Product Launch Agent created with unified market research tool!")

#### Test Real-Time Market Research

In [ ]:
response = agent("Research the current auto loan market - what are the competitive rates and trends?")

#### Test Web Browsing

In [ ]:
response = agent("Browse bankrate.com and analyze their auto loan offerings")

#### Test Marketing Material Generation

In [ ]:
response = agent("Create a marketing poster for a new millennial-focused auto loan product with 5.99% APR")

## 🎉 Lab 1 Complete!

You've successfully created a functional Financial Product Launch Agent prototype! Here's what you accomplished:

- Built an agent with 3 custom tools (product details, market data, web search)
- Tested multi-tool interactions and competitive intelligence gathering
- Established the foundation for our production journey

### Current Limitations (We'll fix these!)
- **Single user conversation memory** - local conversation session, multiple users need multiple sessions
- **Conversation history limited to session** - no long term memory or cross session information
- **Tools reusability** - tools aren't reusable across different agents
- **Running locally only** - not scalable for enterprise use
- **Identity** - No user and/or agent identity or access control
- **Observability** - Limited observability into agent behavior
- **Existing APIs** - No access to existing enterprise APIs for product manager data

##### Next Up: [Lab 2: Personalize our agent by adding memory →](lab-02-add-memory.ipynb)